In [1]:
# ==========================================
# BLOCK 1: DATA INGESTION & CLEANING
# ==========================================
import pandas as pd
import numpy as np
import os

print("Loading UCI E-Commerce Dataset...")
data_path = "../data"
# Note: UCI dataset often contains special characters, so we use unicode_escape
df = pd.read_csv(os.path.join(data_path, "data.csv"), encoding='unicode_escape')

print(f"Initial dataset shape: {df.shape}")

# 1. Drop Missing Customer IDs
# We cannot perform RFM clustering on anonymous transactions
df_clean = df.dropna(subset=['CustomerID']).copy()

# 2. Filter out Returns and Invalid Prices
# Since we are doing pure RFM, we only want positive, successful revenue-generating orders
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# 3. Format the Date Column
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 4. Calculate the Total Amount per line item
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# 5. Clean up the CustomerID format (Pandas loads it as a float by default)
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)

print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Total unique customers: {df_clean['CustomerID'].nunique()}")
display(df_clean.head())

Loading UCI E-Commerce Dataset...
Initial dataset shape: (541909, 8)
Cleaned dataset shape: (397884, 9)
Total unique customers: 4338


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [2]:
# ==========================================
# BLOCK 2: PURE RFM AGGREGATION & CURRENCY CONVERSION
# ==========================================
import pandas as pd

# 1. CURRENCY CONVERSION (GBP to INR)
# We apply a ~105 multiplier so the model trains natively on the INR scale (MAYBE)
df_clean['TotalAmount'] = df_clean['TotalAmount'] * 1

# 2. Define the "Snapshot Date" (The "current" day)
# We take the latest date in the dataset and add 1 day so the most recent buyer has Recency = 1
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# 3. Aggregate R, F, and M for each unique customer
df_rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # RECENCY: Days since last order
    'InvoiceNo': 'nunique',                                   # FREQUENCY: Count of unique orders
    'TotalAmount': 'sum'                                      # MONETARY: Total money spent (in INR)
}).reset_index()

# 4. Rename columns to our standard ML pipeline names
df_rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalAmount': 'Monetary'
}, inplace=True)

# Set CustomerID back as the index for the ML model
df_rfm.set_index('CustomerID', inplace=True)

print(f"RFM Dataset shape: {df_rfm.shape}")
print("\n--- RFM Summary Statistics (INR Scale) ---")
display(df_rfm.describe().round(2))
display(df_rfm.head())

RFM Dataset shape: (4338, 3)

--- RFM Summary Statistics (INR Scale) ---


,Recency,Frequency,Monetary
count,4338.00,4338.00,4338.00
mean,92.54,4.27,2054.27
std,100.01,7.70,8989.23
min,1.00,1.00,3.75
25%,18.00,1.00,307.41
50%,51.00,2.00,674.48
75%,142.00,5.00,1661.74
max,374.00,209.00,280206.02


,Recency,Frequency,Monetary
CustomerID,,,
12346,326,1,77183.60
12347,2,7,4310.00
12348,75,4,1797.24
12349,19,1,1757.55
12350,310,1,334.40


In [3]:
# ==========================================
# BLOCK 3: OUTLIER REMOVAL, SCALING & ELBOW METHOD
# ==========================================
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import joblib
import os

df_math = df_rfm.copy()

# 1. OUTLIER REMOVAL (Dropping the 1% B2B Whales)
# We calculate the 99th percentile cutoff for Frequency and Monetary
f_cutoff = df_math['Frequency'].quantile(0.99)
m_cutoff = df_math['Monetary'].quantile(0.99)

print(f"Dropping customers with > {f_cutoff} orders or > ₹{m_cutoff:.2f} spend.")

# Keep only the bottom 99% of normal customers
df_math = df_math[(df_math['Frequency'] <= f_cutoff) & (df_math['Monetary'] <= m_cutoff)]
print(f"Dataset shape after dropping 1% outliers: {df_math.shape}")

# 2. PURE MATHEMATICAL SCALING (No Log required now!)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_math)
df_scaled = pd.DataFrame(scaled_data, columns=df_math.columns, index=df_math.index)

# Export the Scaler for the Streamlit App
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)
scaler_path = os.path.join(models_dir, "scaler.pkl")
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to {scaler_path}")

# 3. THE ELBOW METHOD
print("Running the Elbow Method...")
inertia_values = []
k_range = range(1, 11)

# Test K from 1 to 10
for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    inertia_values.append(kmeans.inertia_)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia_values, marker='o', linestyle='--', color='b')
plt.title('Elbow Method for Optimal K (UCI Normal Customers)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Distance within clusters)')
plt.xticks(k_range)
plt.grid(True)
plt.show()



Dropping customers with > 30.0 orders or > ₹19881.00 spend.
Dataset shape after dropping 1% outliers: (4275, 3)
Scaler saved to ../models\scaler.pkl
Running the Elbow Method...


<Figure size 1000x600 with 1 Axes>

In [4]:
# ==========================================
# BLOCK 4: TRAIN THE FINAL MODEL & PROFILE CLUSTERS
# ==========================================

print("Training final K-Means model with K=4...")

# 1. Train the model using the 99% filtered and scaled data
optimal_k = 4
final_kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
final_kmeans.fit(df_scaled)

# 2. Predict on the ENTIRE dataset (100% of customers)
# We must scale the full dataset first using the scaler we just trained
full_scaled_data = scaler.transform(df_rfm)
full_scaled_df = pd.DataFrame(full_scaled_data, columns=df_rfm.columns, index=df_rfm.index)

# Predict and attach the cluster labels to the original unscaled RFM dataset
df_rfm['Cluster'] = final_kmeans.predict(full_scaled_df)


# 3. Export the trained model
models_dir = "../models"
model_path = os.path.join(models_dir, "kmeans_model.pkl")
joblib.dump(final_kmeans, model_path)

# 4. Profile the clusters to determine their business meaning
print("Profiling clusters...")
cluster_profile = df_rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
})
# Add customer count per cluster
cluster_profile['Total_Customers'] = df_rfm.groupby('Cluster').size()

# Sort by Monetary value to easily spot the VIPs
cluster_profile = cluster_profile.sort_values(by='Monetary', ascending=False)

print(f"\nModel saved to: {model_path}")
print("\n--- AVERAGE STATS PER CLUSTER ---")
display(cluster_profile.round(2))

Training final K-Means model with K=4...
Profiling clusters...

Model saved to: ../models\kmeans_model.pkl

--- AVERAGE STATS PER CLUSTER ---


,Recency,Frequency,Monetary,Total_Customers
Cluster,,,,
2,17.04,25.05,20349.16,217
3,27.93,8.10,3119.81,711
0,50.32,2.44,768.04,2387
1,251.96,1.48,434.15,1023


In [8]:
# ==========================================
# BLOCK 5: THE INTERACTIVE TESTER
# ==========================================
import pandas as pd
import joblib

# 1. Load the trained assets
scaler = joblib.load("../models/scaler.pkl")
kmeans_model = joblib.load("../models/kmeans_model.pkl")

# ---> SWAP THESE VALUES TO TEST <---
test_recency = 300      # Days since last purchase
test_frequency = 1    # Number of distinct orders
test_monetary = 5100   # Total spend in INR
# -----------------------------------

# 2. Format as DataFrame
test_df = pd.DataFrame([{
    "Recency": test_recency,
    "Frequency": test_frequency,
    "Monetary": test_monetary
}])

# 3. Scale and Predict
expected_columns = scaler.feature_names_in_
test_scaled = scaler.transform(test_df[expected_columns])
predicted_cluster = int(kmeans_model.predict(test_scaled)[0])

print(f"INPUT: Recency={test_recency} | Frequency={test_frequency} | Monetary=₹{test_monetary}")
print(f"PREDICTED CLUSTER ID: {predicted_cluster}")



INPUT: Recency=300 | Frequency=1 | Monetary=₹5100
PREDICTED CLUSTER ID: 1


c:\customer_segmentation\venv\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
